In [102]:

import pandas as pd

In [103]:
ALL_SUBJECTS = [
    "Agriculture", "Geography", "French", "Economics", "Biology", "Igbo", "History", "Physics", "Chemistry",
    "Further Mathematics", "Civic Education", "Computer Science", "Hausa", "English Language", "Government", "Mathematics",
    "Yoruba", "Literature in English"
]

In [104]:
def load_students(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename)

    class_level_map = {
        10: "SS1",
        11: "SS2",
        12: "SS3"
    }

    #  Replaces all values. Any value for the column that's not mapped in the dictionary key, is replace with NaN (meaning an empty value)
    # df["class_level"] = df["class_level"].map(class_level_map)

    # Only replaces matched values. Does not attempt to replace values not provided in the dictionary key.
    df["class_level"] = df["class_level"].replace(class_level_map)

    df["total_score"] = df[ALL_SUBJECTS].mean(axis=1).round(1)

    return df

### Task 1
Task 1
Group the student data by class level and calculate:
- The average score for each subject
- The number oof students in each class level
- The minimum and maximum scores in each subject

In [105]:
def class_statistics(df):
    print("\n" + "="*50)
    print("TASK 1: CLASS LEVEL STATISTICS")
    print("="*50)

    class_level_group_df = df.groupby("class_level")

    # Task 1.1: Average for each subject
    subjects_avg_score = class_level_group_df[ALL_SUBJECTS].mean().round(1).T

    print(subjects_avg_score)

    # Task 1.2: Number of students per class level
    print("\n--- Number of students per class level ---")
    student_counts = class_level_group_df["student_id"].count()
    student_counts.name = "num_students" # this doesnt create a new variable. It creates a label(name) for the Pandas series stored inside the variable.
    #This label name is aonly useful when displaying teh data. SO it is uded when displaying the data

    print(student_counts)

    # Task 1.3: Min & Max scores for all subjects
    print("\n--- Min & max scores (for all subjects) by class level ---")
    min_max = class_level_group_df[ALL_SUBJECTS].agg(["min", "max"])
    print(min_max)

### Task 2
Identify which class level has the:
- Highest average attendance
- Best overall academic performance ( across all subjects)
- Largest gender performance gap in Mathematics


In [ ]:
def class_performance_analysis(df):
    print("\n" + "="*50)
    print("TASK 2: CLASS PERFORMANCE ANALYSIS")
    print("="*50)

    class_level_group_df = df.groupby("class_level")

    # Task 2.1: Highest average attendance, by class level
    avg_attendance = class_level_group_df["attendance"].agg("mean").round(1)
    best_attendance = avg_attendance.idxmax() #idmax returns the index label of the largest value.

    print(avg_attendance)
    print("\n")
    print("="*50)
    print(f"Highest average attendance: Class {best_attendance} ({avg_attendance[best_attendance]}%)")
    # ({avg_attendance[best_attendance]}%) means look up the attendance value for the class stored in best_attendance.

    # Task 2.2: Best overall academic performance, by class level
    print("\n--- Average overall score by class level ---")
    avg_overall = class_level_group_df["total_score"].mean().round(2)
    best_academic_performance = avg_overall.idxmax() # idmax() means return the index label of the largest value
    
    print(f"Best overall performance: Class {best_academic_performance} ({avg_overall[best_academic_performance]}% on avg)")

    # Task 2.3: Largest gender performance gap in Mathematics
    print("\n--- Largest gender performance gap in Mathematics ---")
    gender_maths = df.groupby(["class_level", "gender"])["Mathematics"].mean().round(2)

    print("Gender Maths")
    print(gender_maths)
    print(type(gender_maths))

    gender_maths_wide = gender_maths.unstack(level="gender") #Unstack reshapes the data ... so gender have seperate columns ( M and F)
    gender_maths_wide["gap"] = abs(gender_maths_wide["F"] - gender_maths_wide["M"]).round(2)
    # abs means absolute value of a number - which is its distance from zero, ignring hwether its positive or negative
    # for e.g abs(-7) becomes 7 (thenegative sign disappears)

    biggest_gap = gender_maths_wide["gap"].idxmax()

    print("\n")
    print(gender_maths_wide)
    print(type(gender_maths_wide))
    print(f"\nLargest gender gap in Mathematics can be seen in {biggest_gap} (gap of {gender_maths_wide["gap"][biggest_gap]} points)")

### Task 3
Find out if there's a correlation between attendance and academic performance (hint: use groupby with multiple conditions)

In [ ]:
def attendance_vs_performance(df: pd.DataFrame):
    print("\n" + "="*50)
    print("TASK 3: ATTENDANCE VS ACADEMIC PERFORMANCE")
    print("="*50)

    df["attendance_band"] = pd.cut( #pd.cut() splits continuous numerical values into categories (ranges). In this case, it categorizes attendance percentages. i.e convert numbers into categories
        df["attendance"],
        bins=[0, 70, 85, 100],
        labels=["Low (<=70%)", "Medium (71-85%)", "High (>85%)"]
    )

    attendance_vs_perf = df.groupby(["class_level", "attendance_band"], observed=True)["total_score"].mean().round(2)  
    # observed = true .. only show categories that actually exist in the data. Without it, pandas may show empty combinations of class and attendance bands.

    print(attendance_vs_perf)

    correlation = round(df["attendance"].corr(df["total_score"]), 4)
    #Computes the Pearson correlation coefficient between two variables. This measures how strongly two variables are related.
    # (1.0 - Perfect positive relationship), (0 - No relationship), (-1.0 - Perfect negative relationship)

    print(f"\nCorrelation (attendance vs overall score): {correlation}")
    print("(1.0 = perfect postive correlation/link | 0 = no link | -1.0 = negative link)")

### Task 4
Create a summary report showing the performance metrics for each class level

In [ ]:
def summary_report(df: pd.DataFrame):
    print("\n" + "="*50)
    print("TASK 4: SUMMARY REPORT (PER CLASS LEVEL)")
    print("="*50)

    summary = df.groupby("class_level").agg(
        num_students = ("student_id", "count"), # "student_id" - column to analyze , "count" - function to apply, "num_students" - name of the output column
        avg_attendance = ("attendance", "mean"),
        avg_english = ("English Language", "mean"),
        avg_maths = ("Mathematics", "mean"),
        avg_overall = ("total_score", "mean"),
        min_overall = ("total_score", "min"),
        max_overall = ("total_score", "max")
    ).round(2)

    print(summary.to_string())

    summary.to_csv("../../data/students_report.csv")

In [109]:
def main():
    df = load_students("../../data/students.csv")
    
    class_statistics(df)
    attendance_vs_performance(df)
    class_performance_analysis(df)
    summary_report(df)
   


In [110]:
main()


TASK 1: CLASS LEVEL STATISTICS
class_level             SS1   SS2   SS3
Agriculture            73.5  73.4  73.4
Geography              72.8  73.2  75.2
French                 73.7  72.3  76.7
Economics              74.1  73.9  75.3
Biology                72.6  73.9  71.4
Igbo                   77.8  76.2  68.8
History                74.5  72.9  73.4
Physics                72.3  73.7  73.5
Chemistry              74.5  73.8  74.2
Further Mathematics    73.4  75.5  73.8
Civic Education        73.0  71.1  73.3
Computer Science       74.5  72.2  70.9
Hausa                  69.3  73.0  75.8
English Language       74.3  72.6  74.1
Government             74.5  72.7  75.4
Mathematics            74.9  73.1  73.8
Yoruba                 74.5  69.5  75.8
Literature in English  72.3  72.8  71.7

--- Number of students per class level ---
class_level
SS1    800
SS2    800
SS3    800
Name: num_students, dtype: int64

--- Min & max scores (for all subjects) by class level ---
            Agriculture   